In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import Flowers102
from transformers import AutoImageProcessor
import torchmetrics
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss


In [2]:
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=102,
    ignore_mismatched_sizes=True
)

print(model)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear

In [3]:
# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Unfreeze ONLY the classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

# Verify
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

Trainable: 78,438 / 85,877,094  (0.09%)


In [4]:
MODEL_NAME = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
print(processor)

ViTImageProcessor {
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}



In [5]:
class HFProcessorTransform:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, image):
        inputs = self.processor(image, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0)

transform = HFProcessorTransform(processor)

In [6]:
train_dataset = Flowers102(root="../data", split="train", download=True, transform=transform)
val_dataset   = Flowers102(root="../data", split="val",   download=True, transform=transform)
test_dataset  = Flowers102(root="../data", split="test",  download=True, transform=transform)

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

img, label = train_dataset[0]
print(f"Image shape: {img.shape}")
print(f"Label: {label}")

Train: 1020
Val:   1020
Test:  6149
Image shape: torch.Size([3, 224, 224])
Label: 0


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-4,             # higher LR — we're only training a fresh head
    weight_decay=0.01
)
criterion = CrossEntropyLoss()

In [8]:
from tqdm.auto import tqdm

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0., 0, 0

    for batch in tqdm(loader, desc="Train"):
        pixel_values = batch[0].to(device)
        labels = batch[1].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(loader, desc="Eval"):
        pixel_values = batch[0].to(device)
        labels = batch[1].to(device)

        outputs = model(pixel_values=pixel_values).logits
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100 * correct / total

In [10]:
EPOCHS = 5
history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    print(f"Epoch {epoch + 1}/{EPOCHS}, "
          f"Train Loss: {tr_loss:.4f}, Acc {tr_acc:.4f}% | "
          f"Val Loss: {va_loss:.4f}, Acc {va_acc:.4f}%")

Train:   0%|          | 0/32 [00:00<?, ?it/s]

Eval:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1/5, Train Loss: 3.8590, Acc 95.6863% | Val Loss: 3.6019, Acc 97.3529%


Train:   0%|          | 0/32 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd06b448400>
Traceback (most recent call last):
  File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd06b448400>
Traceback (most recent call last):
  File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/datal

...

Epoch 4/5, Train Loss: 2.3671, Acc 99.4118% | Val Loss: 2.2081, Acc 98.2353%


Train:   0%|          | 0/32 [00:00<?, ?it/s]

Eval:   0%|          | 0/32 [00:00<?, ?it/s]

Exception ignored in:   File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7dd06b448400>
Traceback (most recent call last):
  File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()Exception ignored in: 
  File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7dd06b448400>
    if w.is_alive():Traceback (most recent call last):

      self._shutdown_workers() 
   File "/home/harish_kumar/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
      if w.is_alive(): 
^ ^ ^ ^ ^ ^ ^ ^^^^^^^^^^
  File "/home/harish_kumar/miniconda3/envs

Epoch 5/5, Train Loss: 1.9433, Acc 99.5098% | Val Loss: 1.8220, Acc 98.2353%


In [11]:
torch.save(model.state_dict(), "../saved_models/vit_head_only.pth")